# Проверка визуализаций: GT и предикты

Ноутбук проверяет весь стек визуализации: `viz.compare` и `viz.errors`.

| Секция | Нужен датасет? | Нужна модель? |
|--------|----------------|---------------|
| 1. Синтетические данные | нет | нет |
| 2. GT-боксы на изображении | нет | нет |
| 3. GT vs Pred side-by-side | нет | нет (синт. предикты) |
| 4. compare_gt_pred | желателен | **да** |
| 5. Error analysis (FP/FN) | нет | нет (синт. предикты) |
| 6. Error analysis с моделью | **да** | **да** |

In [ ]:
import sys
from pathlib import Path

# позволяет запускать ноутбук из любой директории
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## 1. Конфигурация путей

Заполни переменные ниже. Если путей нет — всё равно запускай, везде есть синтетический фоллбэк.

In [ ]:
import os

# --- пути к данным ---
# Можно задать через env-переменную DATA_ROOT или напрямую здесь
DATA_ROOT = Path(os.environ.get("DATA_ROOT", ""))

# Путь к изображениям val-сплита (ожидается рядом папка labels/)
IMAGES_DIR = DATA_ROOT / "images" / "val" if DATA_ROOT.exists() else None

# Путь к .pt модели
MODEL_PATH = None  # например: Path("runs/train/exp/weights/best.pt")

# Имена классов — перечисли свои или оставь пример
CLASS_NAMES = ["cat", "dog", "person"]

# Путь к data.yaml (альтернатива ручному списку CLASS_NAMES)
DATA_YAML = None  # например: Path("configs/data.yaml")
if DATA_YAML and Path(DATA_YAML).exists():
    from cv_lib.data import class_names_from_yaml
    CLASS_NAMES = class_names_from_yaml(DATA_YAML)
    print(f"Классы из yaml: {CLASS_NAMES}")

print(f"DATA_ROOT  : {DATA_ROOT  or '(не задан)'}")
print(f"IMAGES_DIR : {IMAGES_DIR or '(не задан)'}")
print(f"MODEL_PATH : {MODEL_PATH or '(не задан)'}")
print(f"CLASS_NAMES: {CLASS_NAMES}")

## 2. Синтетические тестовые данные

Создаём временную директорию с изображениями и YOLO-лейблами — чтобы следующие секции работали без реального датасета.

In [ ]:
import tempfile

_tmp = Path(tempfile.mkdtemp(prefix="cv_lib_viz_test_"))
SYNTH_IMAGES = _tmp / "images" / "val"
SYNTH_LABELS = _tmp / "labels" / "val"
SYNTH_PRED_LABELS = _tmp / "pred_labels"
for d in (SYNTH_IMAGES, SYNTH_LABELS, SYNTH_PRED_LABELS):
    d.mkdir(parents=True, exist_ok=True)


def _make_image(w: int = 640, h: int = 480, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    img = rng.integers(40, 200, (h, w, 3), dtype=np.uint8)
    # нарисуем цветные прямоугольники как «объекты»
    for i, (x, y, color) in enumerate([
        (80, 60, (0, 200, 80)),
        (350, 150, (200, 80, 0)),
        (500, 300, (80, 0, 200)),
    ]):
        cv2.rectangle(img, (x, y), (x + 120, y + 100), color, -1)
    return img


# GT лейблы: каждый объект — class cx cy w h (YOLO нормализованные)
_GT = [
    # name,  cx,    cy,    bw,    bh
    ("img0", [(0, 0.218, 0.208, 0.187, 0.208),   # кот слева
              (1, 0.664, 0.406, 0.187, 0.208),   # пёс в центре
              (2, 0.875, 0.677, 0.187, 0.208)]), # человек справа
    ("img1", [(0, 0.218, 0.208, 0.187, 0.208),
              (2, 0.875, 0.677, 0.187, 0.208)]),
    ("img2", [(1, 0.664, 0.406, 0.187, 0.208)]),
]

# Синтетические предикты: немного сдвинуты + один лишний FP, один GT пропущен
_PRED = [
    ("img0", [(0, 0.230, 0.220, 0.187, 0.208, 0.91),  # TP кот
              (1, 0.680, 0.420, 0.187, 0.208, 0.78),  # TP пёс
              (0, 0.550, 0.550, 0.100, 0.120, 0.35)]),# FP кот
    ("img1", [(0, 0.230, 0.220, 0.187, 0.208, 0.88)]),# FN: нет второго класса
    ("img2", [(1, 0.680, 0.420, 0.187, 0.208, 0.65),
              (2, 0.200, 0.200, 0.100, 0.100, 0.29)]),# FP человек
]

for seed, (name, gt_boxes) in enumerate(_GT):
    img = _make_image(seed=seed)
    cv2.imwrite(str(SYNTH_IMAGES / f"{name}.jpg"), img)
    gt_lines = [f"{c} {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f}" for c, cx, cy, bw, bh in gt_boxes]
    (SYNTH_LABELS / f"{name}.txt").write_text("\n".join(gt_lines))

for name, pred_boxes in _PRED:
    pred_lines = [f"{c} {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f} {conf:.4f}"
                  for c, cx, cy, bw, bh, conf in pred_boxes]
    (SYNTH_PRED_LABELS / f"{name}.txt").write_text("\n".join(pred_lines))

print(f"Синтетические данные: {SYNTH_IMAGES}")
print(f"  изображений: {len(list(SYNTH_IMAGES.glob('*.jpg')))}")
print(f"  GT лейблов : {len(list(SYNTH_LABELS.glob('*.txt')))}")
print(f"  pred лейблов: {len(list(SYNTH_PRED_LABELS.glob('*.txt')))}")

## 3. GT-боксы на одном изображении

Проверяем `load_yolo_gt` и внутренний `_draw_boxes` — базовый рендеринг без модели.

In [ ]:
from cv_lib.viz.compare import _draw_boxes, load_yolo_gt

# Берём первое изображение из синтетического набора (или реального, если задан)
images_dir = IMAGES_DIR if (IMAGES_DIR and IMAGES_DIR.exists()) else SYNTH_IMAGES
labels_dir = images_dir.parent.parent / "labels" / images_dir.name

sample_images = sorted(images_dir.glob("*.jpg")) + sorted(images_dir.glob("*.png"))
if not sample_images:
    raise FileNotFoundError(f"Нет изображений в {images_dir}")

img_path = sample_images[0]
label_path = (labels_dir / img_path.stem).with_suffix(".txt")

img_bgr = cv2.imread(str(img_path))
h, w = img_bgr.shape[:2]
gt_boxes, gt_cls = load_yolo_gt(label_path, w, h)

print(f"Изображение : {img_path.name}  ({w}×{h})")
print(f"GT боксов   : {len(gt_boxes)}")
for i, (box, cid) in enumerate(zip(gt_boxes, gt_cls)):
    cname = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
    print(f"  [{i}] class={cname}  xyxy={box.astype(int).tolist()}")

vis = _draw_boxes(img_bgr, gt_boxes, gt_cls, CLASS_NAMES)
rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(rgb)
ax.set_title(f"GT: {img_path.name}  |  {len(gt_boxes)} боксов")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. GT vs предикты — сетка по нескольким изображениям

Читаем GT и сохранённые предикты (YOLO txt), рисуем панели рядом без запуска модели.
Полезно для быстрой проверки после `yolo detect predict ... save_txt=True`.

In [ ]:
from cv_lib.viz.compare import _add_panel_label, _draw_boxes, load_yolo_gt

pred_labels_dir = SYNTH_PRED_LABELS  # замени на реальную папку с предиктами

MAX_IMAGES = 4
pairs = []
for img_path in sorted(images_dir.glob("*.jpg"))[:MAX_IMAGES]:
    gt_lbl = (labels_dir / img_path.stem).with_suffix(".txt")
    pred_lbl = (pred_labels_dir / img_path.stem).with_suffix(".txt")
    pairs.append((img_path, gt_lbl, pred_lbl))


def _load_pred_txt(path: Path, w: int, h: int):
    """Читает YOLO pred txt с опциональной колонкой conf."""
    if not path.exists():
        return np.zeros((0, 4), dtype=np.float32), np.zeros(0, int), np.zeros(0, np.float32)
    boxes, cls, confs = [], [], []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:5])
        conf = float(parts[5]) if len(parts) >= 6 else 1.0
        x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
        x2, y2 = (cx + bw / 2) * w, (cy + bh / 2) * h
        boxes.append([x1, y1, x2, y2]); cls.append(cid); confs.append(conf)
    b = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
    return b, np.array(cls, int), np.array(confs, np.float32)


panels = []
for img_path, gt_lbl, pred_lbl in pairs:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    gt_boxes, gt_cls = load_yolo_gt(gt_lbl, w, h)
    pred_boxes, pred_cls, pred_conf = _load_pred_txt(pred_lbl, w, h)

    gt_panel = _draw_boxes(img, gt_boxes, gt_cls, CLASS_NAMES)
    _add_panel_label(gt_panel, f"GT ({len(gt_boxes)})")

    pred_panel = _draw_boxes(img, pred_boxes, pred_cls, CLASS_NAMES, scores=pred_conf)
    _add_panel_label(pred_panel, f"Pred ({len(pred_boxes)})")

    divider = np.full((h, 4, 3), 80, dtype=np.uint8)
    row = np.concatenate([gt_panel, divider, pred_panel], axis=1)
    panels.append((img_path.name, row))

fig, axes = plt.subplots(len(panels), 1, figsize=(14, 5 * len(panels)))
if len(panels) == 1:
    axes = [axes]
for ax, (name, row) in zip(axes, panels):
    ax.imshow(cv2.cvtColor(row, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("GT (слева)  vs  Pred из txt (справа)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. compare_gt_pred — запуск модели прямо в ноутбуке

Требует реальный `.pt`-файл в `MODEL_PATH`. Пропускается, если модели нет.

In [ ]:
from cv_lib.viz.compare import compare_gt_pred

if MODEL_PATH is None or not Path(MODEL_PATH).exists():
    print("MODEL_PATH не задан или файл не найден — ячейка пропущена.")
    print("Задай MODEL_PATH в ячейке конфигурации и перезапусти.")
else:
    img_path = sample_images[0]
    result = compare_gt_pred(
        image_path=img_path,
        model_path=MODEL_PATH,
        class_names=CLASS_NAMES,
        conf_threshold=0.25,
        show=True,   # автоматически использует matplotlib в ноутбуке
    )
    print(f"Результат: {result.shape}  (GT слева, предикты справа)")

## 6. Анализ ошибок: FP / FN по датасету

`find_errors` принимает либо модель, либо папку с готовыми предиктами.  
Здесь используем синтетические предикты — модель не нужна.

In [ ]:
from cv_lib.viz.errors import find_errors

errors = find_errors(
    images_dir=images_dir,
    labels_dir=labels_dir,
    pred_labels_dir=pred_labels_dir,  # замени на None + model_path=MODEL_PATH для живого инференса
    conf_threshold=0.25,
    iou_threshold=0.50,
)

fp = [e for e in errors if e.error_type == "FP"]
fn = [e for e in errors if e.error_type == "FN"]

print(f"Всего ошибок : {len(errors)}")
print(f"  FP (лишние предикты) : {len(fp)}")
print(f"  FN (пропущенные GT)  : {len(fn)}")
print()
for e in errors:
    cname = CLASS_NAMES[e.class_id] if e.class_id < len(CLASS_NAMES) else str(e.class_id)
    conf_str = f"conf={e.conf:.2f}" if e.error_type == "FP" else "gt only"
    print(f"  {e.error_type}  {e.image_path.name:<12}  class={cname:<8}  {conf_str}")

## 7. Сетка ошибок

`render_errors` — тайлы с FP (красный) и FN (синий).

In [ ]:
from cv_lib.viz.errors import render_errors

if not errors:
    print("Ошибок не найдено — сетка пустая.")
else:
    grid = render_errors(
        entries=errors,
        class_names=CLASS_NAMES,
        tile_size=(280, 280),
        cols=4,
        max_tiles=16,
        show=True,   # рендерит инлайн через matplotlib
        # output_path="errors_grid.jpg",  # раскомментируй чтобы сохранить
    )
    print(f"Сетка: {grid.shape[1]}×{grid.shape[0]} пкс, {len(errors)} тайлов")

## 8. Анализ ошибок с живой моделью

Требует `MODEL_PATH` и реальный датасет в `IMAGES_DIR`.

In [ ]:
if MODEL_PATH is None or not Path(MODEL_PATH).exists():
    print("MODEL_PATH не задан — ячейка пропущена.")
elif IMAGES_DIR is None or not IMAGES_DIR.exists():
    print("IMAGES_DIR не задан — ячейка пропущена.")
else:
    live_errors = find_errors(
        images_dir=IMAGES_DIR,
        model_path=MODEL_PATH,
        conf_threshold=0.25,
        iou_threshold=0.50,
    )
    print(f"FP: {sum(1 for e in live_errors if e.error_type == 'FP')}  "
          f"FN: {sum(1 for e in live_errors if e.error_type == 'FN')}")

    render_errors(
        entries=live_errors,
        class_names=CLASS_NAMES,
        tile_size=(320, 320),
        cols=4,
        max_tiles=24,
        show=True,
        output_path=DATA_ROOT / "error_grid.jpg" if DATA_ROOT.exists() else None,
    )

## 9. Очистка временных файлов

In [ ]:
import shutil

shutil.rmtree(_tmp, ignore_errors=True)
print(f"Удалено: {_tmp}")